In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import os
import joblib
import torch
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics
from src.utils.inference_metrics import compute_predictive_entropy

CONTEXTS_DATASET_PATH = "../../../../data/squadv2/contexts.csv"
QA_DATASET_PATH = "../../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../../models/Qwen/Qwen2.5-7B-Instruct" # "Undi95/Meta-Llama-3-8B-Instruct-hf" / "../../../../models/Qwen/Qwen2.5-7B-Instruct"

/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PARAMS = {
    'version': "2.1.2",
    'num_samples': 2000,
    'num_contexts': 5,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. If there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024, 'do_sample': False, 'num_beams': 1},
    'stub_answer': "I do not have an answer to your question",
    'calculate_entropy': True,
    'revert': True,
    'centered': False
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs_v2'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

Creating Dir...


### Подключение к агенту

In [3]:
agent = CustomAgent(PARAMS['model'], output_logits=PARAMS['calculate_entropy'], use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards: 100%|██████████| 7/7 [00:09<00:00,  1.30s/it]
/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `to

The question of what is "wrong" with humanity is complex and multifaceted, as it can be interpreted in various ways depending on the context. Here are some common perspectives:

1. **Inequality and Discrimination**: Human societies often struggle with issues of inequality based on race, gender, socioeconomic status, and other factors.

2. **Violence and Conflict**: Despite advancements in technology and communication, human societies continue to experience violence, war, and conflict.

3. **Environmental Degradation**: There's a growing concern about how human activities are affecting the environment, leading to climate change, pollution, and loss of biodiversity.

4. **Moral and Ethical Dilemmas**: Humans often face difficult moral and ethical decisions that can lead to conflicts and suffering.

5. **Psychological Issues**: Mental health problems such as depression, anxiety, and addiction are prevalent and can significantly impact individuals and society.

6. **Economic Inequity**: We

### Формируем список контекстов для каждого запроса со скорами

In [4]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [5]:
contexts_df = pd.read_csv(CONTEXTS_DATASET_PATH)

In [6]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_rel_id = dataset_df['relevant_context_id'][i]
    cur_list_ids = [(-1, cur_rel_id)]

    while len(cur_list_ids) != PARAMS['num_contexts']:
        unrel_context_id = random.randint(0, contexts_df.shape[0]-1)

        prep_cntx = (-1, unrel_context_id)
        if unrel_context_id != cur_rel_id:
            cur_list_ids.append(prep_cntx)

    # shuffling strategy
    if PARAMS['revert']:
        cur_list_ids = cur_list_ids[::-1]
    elif PARAMS['centered']:
        cur_list_ids.pop(0)
        cur_list_ids.insert(len(cur_list_ids)//2, (-1, cur_rel_id))
    
    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 2000/2000 [00:00<00:00, 143598.75it/s]


In [7]:
print(CONTEXTS_LIST_IDS[0])

[(-1, 8024), (-1, 9012), (-1, 819), (-1, 3648), (-1, np.int64(0))]


### Готовим промпт

In [8]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    docs = [contexts_df['context'][CONTEXTS_LIST_IDS[i][j][1]] for j in range(len(CONTEXTS_LIST_IDS[i]))]
    documents_list = [PARAMS['item_format'].format(document=doc.strip()) for doc in docs]
    
    documents_list = '\n'.join(documents_list)
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 2000/2000 [00:00<00:00, 70864.69it/s]


In [9]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [10]:
del contexts_df
gc.collect()

66

In [11]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. If there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- Tourism accounts for much of Uruguay's economy. Tourism in Montevideo is centered in the Ciudad Vieja area, which includes the city's oldest buildings, several museums, art galleries, and nightclubs, with Sarandí Street and the Mercado del Puerto being the most frequented venues of the old city. On the edge of Ciudad Vieja, Plaza Independencia is surrounded by many sights, including the Solís Theatre and the Palacio Salvo; the plaza also constitutes one end of 18 de Julio Avenue, the city's most important tourist destination

### Генерируем ответы на вопросы

In [12]:
generate_answers, calc_metrics = [], []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])

    cur_metrics = dict()
    if PARAMS['calculate_entropy']:
        logits = torch.cat(meta_info['logits'], 0).cpu().detach()
        entropy = compute_predictive_entropy(logits)
        cur_metrics['predictive_entropy'] = float(entropy)
    calc_metrics.append(cur_metrics)
    generate_answers.append(pred_answer)
    
    # logits = torch.cat(meta_info['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}\nMETRICS: {cur_metrics}")
e_time = time()

/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/opt/conda/envs/smiles_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
  0%|          | 1/2000 [00:00<30:37,  1.09it/s]


[0]: 
GEN: Beyoncé started becoming popular in the late 1990s as the lead singer of R&B girl-group Destiny's Child.
GOLD: in the late 1990s
METRICS: {'predictive_entropy': 0.8320853114128113}


  5%|▌         | 101/2000 [00:41<13:15,  2.39it/s]


[100]: 
GEN: eleven
GOLD: eleven
METRICS: {'predictive_entropy': 0.5905574560165405}


 10%|█         | 201/2000 [01:18<12:16,  2.44it/s]

...

 90%|█████████ | 1801/2000 [12:25<01:04,  3.07it/s]


[1800]: 
GEN: Kane Kramer called the device the IXI.
GOLD: IXI
METRICS: {'predictive_entropy': 0.6049479246139526}


 95%|█████████▌| 1901/2000 [13:02<00:39,  2.53it/s]


[1900]: 
GEN: September 12, 2006
GOLD: September 12, 2006
METRICS: {'predictive_entropy': 0.08041352033615112}


100%|██████████| 2000/2000 [13:38<00:00,  2.44it/s]


In [13]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), int(item[1])) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {
        'gen_answer': str(generate_answers[i]), 
        'metainfo': calc_metrics[i], 
        'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [14]:
LOADING_VERSION = "3.1.1"

In [15]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [16]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [17]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [18]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [19]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [05:06<00:00,  6.52it/s, BLEU2=0.0252, BLEU1=0.0285, ExactMatch=0.0286, METEOR=0.0329, BertScore=nan, Levenshtain=38.8, ROUGEL=0.0334]


In [20]:
LOADING_VERSION

'3.1.1'

In [21]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))